# Aiscern — Video Deepfake Frame Detector Fine-Tuning
**Platform:** Google Colab (T4 GPU)  
**Target:** `saghi776/aiscern-video-detector`  
**Base model:** `google/vit-base-patch16-224-in21k`  
**Runtime:** ~60 minutes  
**Expected accuracy:** >82% on FaceForensics++ test frames

### Before running:
1. Runtime → Change runtime type → **T4 GPU**
2. Click 🔑 key icon (left sidebar) → Add secret: `HF_TOKEN`
3. Create `saghi776/aiscern-video-detector` repo on huggingface.co/new

> **⚠️ Colab disconnects training!** This notebook saves checkpoints to Google Drive automatically.

In [ ]:
# CELL 1 — Mount Google Drive for checkpoint persistence
# If Colab disconnects, re-run from CELL 8 and training resumes from last checkpoint
from google.colab import drive
drive.mount('/content/drive')
CHECKPOINT_DIR = '/content/drive/MyDrive/aiscern-video-checkpoints'
import os; os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f'✅ Checkpoints will be saved to: {CHECKPOINT_DIR}')

In [ ]:
# CELL 2 — Install dependencies
!pip install -q transformers==4.40.0 datasets accelerate huggingface_hub \
    pillow scikit-learn evaluate

In [ ]:
# CELL 3 — Authenticate HuggingFace
from google.colab import userdata
from huggingface_hub import login

HF_TOKEN = userdata.get('HF_TOKEN')
login(token=HF_TOKEN, add_to_git_credential=False)
print('✅ Authenticated with HuggingFace')

In [ ]:
# CELL 4 — Configuration
import torch

BASE_MODEL  = 'google/vit-base-patch16-224-in21k'
REPO_ID     = 'saghi776/aiscern-video-detector'
BATCH_SIZE  = 16    # smaller for Colab 15GB VRAM
EPOCHS      = 4
LR          = 2e-5
SEED        = 42
IMG_SIZE    = 224
MAX_SAMPLES = 16000  # 8K real + 8K deepfake

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# CELL 5 — Load deepfake face datasets
# These are frame-level image datasets (faces from deepfake videos)
# Labels: 0 = REAL face frame, 1 = DEEPFAKE face frame

from datasets import load_dataset, concatenate_datasets
from PIL import Image as PILImage
import numpy as np

all_splits = []

def normalise_label(val):
    s = str(val).upper().strip()
    if s in ('FAKE','AI','1','DEEPFAKE','GENERATED','SYNTHETIC','MANIPULATED'): return 1
    if s in ('REAL','HUMAN','0','AUTHENTIC','GENUINE','ORIGINAL'):              return 0
    return -1

def validate_labels(ds, name):
    """Hard check: reject datasets with wrong label format."""
    sample = [normalise_label(ds[i]['label']) for i in range(min(20, len(ds)))]
    if -1 in sample or len(set(sample)) < 2:
        print(f'  ❌ {name}: unexpected labels — skipping')
        return False
    return True

# 1. DeepFake vs Real Faces (primary)
for ds_id in [
    'arnabdhar/DeepFake-Vs-Real-Faces',
    'marcelomoreno26/deepfake-detection',
    'jlbaker361/cifake-real-and-ai-generated-small-images',  # fallback
]:
    try:
        print(f'Loading {ds_id}...')
        ds = load_dataset(ds_id, split='train', token=HF_TOKEN)

        img_col = next((c for c in ds.column_names
                        if c.lower() in ('image','img','face','photo')), None)
        lbl_col = next((c for c in ds.column_names
                        if 'label' in c.lower()), None)

        if not img_col or not lbl_col:
            print(f'  ⚠️  Missing columns in {ds_id}')
            continue

        if img_col != 'image':
            ds = ds.rename_column(img_col, 'image')

        ds = ds.map(lambda x: {'label': normalise_label(x[lbl_col])})
        ds = ds.filter(lambda x: x['label'] != -1)
        ds = ds.select_columns(['image', 'label'])

        if not validate_labels(ds, ds_id):
            continue

        real = ds.filter(lambda x: x['label']==0).num_rows
        fake = ds.filter(lambda x: x['label']==1).num_rows

        if real > 100 and fake > 100:
            all_splits.append(ds)
            print(f'  ✅ {ds_id.split("/")[-1]}: {len(ds):,} (real={real:,} fake={fake:,})')
    except Exception as e:
        print(f'  ⚠️  {ds_id.split("/")[-1]} skipped: {e}')

if not all_splits:
    raise RuntimeError('No datasets loaded — check HF_TOKEN and internet')

combined = concatenate_datasets(all_splits)
print(f'\nCombined: {len(combined):,} samples')

In [ ]:
# CELL 6 — Balance and split
real_ds  = combined.filter(lambda x: x['label'] == 0).shuffle(SEED)
fake_ds  = combined.filter(lambda x: x['label'] == 1).shuffle(SEED)
n        = min(len(real_ds), len(fake_ds), MAX_SAMPLES // 2)

# Final label safety check
assert set(combined.unique('label')) == {0, 1}, 'Labels must be exactly {0, 1}'

balanced = concatenate_datasets([
    real_ds.select(range(n)),
    fake_ds.select(range(n)),
]).shuffle(SEED)

split    = balanced.train_test_split(test_size=0.15, seed=SEED)
test_ds  = split['test']
val_split = split['train'].train_test_split(test_size=0.1, seed=SEED)
train_ds  = val_split['train']
val_ds    = val_split['test']

print(f'Train: {len(train_ds):,}  Val: {len(val_ds):,}  Test: {len(test_ds):,}')

In [ ]:
# CELL 7 — Preprocessing
from transformers import ViTImageProcessor
from PIL import Image as PILImage
import random
from PIL import ImageEnhance

processor = ViTImageProcessor.from_pretrained(BASE_MODEL)

def to_pil(img):
    if isinstance(img, PILImage.Image): return img.convert('RGB')
    return PILImage.fromarray(np.uint8(img)).convert('RGB')

def augment_train(img):
    img = to_pil(img).resize((IMG_SIZE, IMG_SIZE), PILImage.LANCZOS)
    if random.random() > 0.5: img = img.transpose(PILImage.FLIP_LEFT_RIGHT)
    if random.random() > 0.5: img = ImageEnhance.Brightness(img).enhance(random.uniform(0.85, 1.15))
    if random.random() > 0.5: img = ImageEnhance.Contrast(img).enhance(random.uniform(0.85, 1.15))
    return img

def preprocess_train(examples):
    imgs = [augment_train(i) for i in examples['image']]
    out  = processor(images=imgs, return_tensors='np')
    out['labels'] = examples['label']
    return out

def preprocess_eval(examples):
    imgs = [to_pil(i).resize((IMG_SIZE, IMG_SIZE)) for i in examples['image']]
    out  = processor(images=imgs, return_tensors='np')
    out['labels'] = examples['label']
    return out

print('Preprocessing (may take a few minutes)...')
train_ds = train_ds.map(preprocess_train, batched=True, batch_size=128,
                         remove_columns=['image'], desc='Train')
val_ds   = val_ds.map(preprocess_eval,   batched=True, batch_size=128,
                       remove_columns=['image'], desc='Val')
test_ds  = test_ds.map(preprocess_eval,  batched=True, batch_size=128,
                        remove_columns=['image'], desc='Test')

train_ds.set_format('torch'); val_ds.set_format('torch'); test_ds.set_format('torch')
print('✅ Preprocessing done')

In [ ]:
# CELL 8 — Load model
# id2label: 0='REAL', 1='DEEPFAKE'
# hf-analyze.ts: /ai|fake|deepfake|generated/i matches 'DEEPFAKE' ✅
#                /real|authentic|genuine/i    matches 'REAL'      ✅
from transformers import ViTForImageClassification

model = ViTForImageClassification.from_pretrained(
    BASE_MODEL,
    num_labels=2,
    id2label={0: 'REAL', 1: 'DEEPFAKE'},
    label2id={'REAL': 0, 'DEEPFAKE': 1},
    ignore_mismatched_sizes=True,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable params: {trainable/1e6:.1f}M')

In [ ]:
# CELL 9 — Training with Colab-safe checkpoint strategy
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback
from sklearn.metrics import accuracy_score, f1_score
import numpy as np

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy': float(accuracy_score(labels, preds)),
        'f1':       float(f1_score(labels, preds, average='binary')),
    }

training_args = TrainingArguments(
    output_dir=CHECKPOINT_DIR,      # Saves to Drive — survives disconnects
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LR,
    warmup_ratio=0.1,
    weight_decay=0.01,
    evaluation_strategy='steps',
    eval_steps=200,                 # Evaluate every 200 steps
    save_strategy='steps',
    save_steps=200,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    greater_is_better=True,
    logging_steps=50,
    fp16=True,
    push_to_hub=False,              # Push manually at end
    report_to='none',
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

# Resume from checkpoint if Colab disconnected
last_checkpoint = None
if os.path.exists(CHECKPOINT_DIR):
    checkpoints = sorted([d for d in os.listdir(CHECKPOINT_DIR)
                          if d.startswith('checkpoint')])
    if checkpoints:
        last_checkpoint = os.path.join(CHECKPOINT_DIR, checkpoints[-1])
        print(f'Resuming from: {last_checkpoint}')

print(f'Training on {len(train_ds):,} samples...')
trainer.train(resume_from_checkpoint=last_checkpoint)

In [ ]:
# CELL 10 — Evaluate
print('Evaluating on test set...')
test_results = trainer.evaluate(test_ds)

print(f"\n{'='*40}")
print(f"TEST ACCURACY: {test_results['eval_accuracy']:.4f}  ({test_results['eval_accuracy']*100:.1f}%)")
print(f"TEST F1:       {test_results['eval_f1']:.4f}")
print(f"{'='*40}")

if test_results['eval_accuracy'] >= 0.82:
    print('✅ Target accuracy >82% achieved!')
else:
    print('⚠️  Below 82% target — consider more epochs')

In [ ]:
# CELL 11 — Push to HuggingFace
# Run this cell even after a disconnect — model loads from Drive checkpoint
commit_msg = (
    f"ViT deepfake frame classifier — "
    f"accuracy={test_results['eval_accuracy']:.3f} "
    f"f1={test_results['eval_f1']:.3f}"
)
model.push_to_hub(REPO_ID, commit_message=commit_msg, token=HF_TOKEN)
processor.push_to_hub(REPO_ID, token=HF_TOKEN)
print(f'\n✅ Model pushed to: https://huggingface.co/{REPO_ID}')

In [ ]:
# CELL 12 — Verify output format
# hf-analyze.ts: /ai|fake|deepfake|generated/i → DEEPFAKE ✅
#                /real|authentic|genuine/i      → REAL     ✅
from transformers import pipeline
from PIL import Image as PILImage
import numpy as np

print('Verifying inference output format...')
pipe = pipeline('image-classification', model=REPO_ID, token=HF_TOKEN,
                device=0 if torch.cuda.is_available() else -1)

# Test with a solid-colour image (clearly not a real photo)
test_img = PILImage.fromarray(np.zeros((224, 224, 3), dtype=np.uint8))
result   = pipe(test_img)
print(f'\nInference output:')
print(result)
print(f'\nExpected: [{{"label": "DEEPFAKE", "score": X}}, {{"label": "REAL", "score": Y}}]')

labels_out = [r['label'] for r in result]
assert 'DEEPFAKE' in labels_out, f'ERROR: DEEPFAKE missing! Got: {labels_out}'
assert 'REAL'     in labels_out, f'ERROR: REAL missing! Got: {labels_out}'
print('\n✅ Output format correct — compatible with hf-analyze.ts')